# 1 · Static filters (the baseline)

Before training anything: the upscalers that need no model, the resampling filters every image library has. They are
scored on notebook 0's validation split exactly as every model is (`training.val_scores`: SSIM-Y and luma PSNR of each
patch's core), and recorded for notebook 29's table of every notebook's results.

`models.bicubic`, torch's bicubic (a = −0.75), is the skip every model here is built on, and the bar each model's gain
is measured against.

At the end, the three filters ONNX can express as one `Resize` node (nearest, linear, cubic) are exported at the
client's 320×180 step and as a whole 1080p frame, for the browser benchmark. A filter has no weights, so float32 only.

Needs: `00_data_preprocess.ipynb`.

In [1]:
"""Settings."""

import os

import numpy as np
import torch
from PIL import Image

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, training, webexport
from helpers.training import DEVICE

NOTEBOOK = 1
STEP_WIDTH, STEP_HEIGHT = 320, 180       # LR pixels one client tile adds to the picture (notebook 5)
FRAME_WIDTH, FRAME_HEIGHT = 1920, 1080   # the whole-frame graph

manifest = training.load_manifest()
SCALE = manifest["scale"]
print(f"validation: {manifest['splits']['val']['patches']:,} patches; device {DEVICE}")

validation: 24,286 patches; device cuda


## 1. The filters

Ordered by how far they reach:

- **nearest** copies the closest pixel: blocky, but invents nothing;
- **bilinear** averages the four nearest;
- **bicubic** fits a cubic through four pixels a side, which sharpens and can ring at hard edges. torch's (a = −0.75,
  the models' skip) and PIL's (a = −0.5, softer);
- **Lanczos-3** reaches three pixels a side, sharper still.

The PIL filters run in float on each colour, so they are not rounded to 8 bits on the way.

In [2]:
def resize(mode):
    module = models.Resize(SCALE, mode).to(DEVICE)
    return lambda lr, condition: module(lr)


def pil(resample):
    """A PIL filter, in float, on each colour of each patch."""
    def upscale(lr, condition):
        x = lr.cpu().numpy()
        n, colours, h, w = x.shape
        out = np.empty((n, colours, h * SCALE, w * SCALE), np.float32)
        for i in range(n):
            for c in range(colours):
                out[i, c] = np.asarray(Image.fromarray(x[i, c]).resize((w * SCALE, h * SCALE), resample))
        return torch.from_numpy(out).to(lr.device)
    return upscale


FILTERS = {
    "nearest": resize("nearest"),
    "bilinear": resize("bilinear"),
    "bicubic": resize("bicubic"),
    "bicubic, PIL (a = -0.5)": pil(Image.BICUBIC),
    "Lanczos-3, PIL": pil(Image.LANCZOS),
}

results = {name: training.val_scores(upscale, manifest) for name, upscale in FILTERS.items()}
for name, scores in results.items():
    training.record(NOTEBOOK, name, scores)

## 2. Scores

SSIM-Y, and luma and chroma PSNR, over every validation patch; the gain over bicubic with its 95% range from a bootstrap over the
sources, and by kind and encoder line. Then the gain by slice QP.

In [3]:
training.show(training.compare(results, manifest, base=results["bicubic"]))

                  model SSIM-Y over bicubic          95% range luma dB dB over chroma dB over   video  screen   photo    x264   nvenc
         Lanczos-3, PIL 0.8528      +0.0007 +0.0006 to +0.0009   28.70   +0.12          +0.00 -0.0000 +0.0013 +0.0007 +0.0001 +0.0014
                bicubic 0.8521      +0.0000 +0.0000 to +0.0000   28.58   +0.00          +0.00 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000
bicubic, PIL (a = -0.5) 0.8513      -0.0008 -0.0010 to -0.0006   28.50   -0.08          +0.01 +0.0005 -0.0017 -0.0013 -0.0001 -0.0015
               bilinear 0.8434      -0.0087 -0.0096 to -0.0078   27.97   -0.61          +0.01 -0.0018 -0.0140 -0.0102 -0.0049 -0.0124
                nearest 0.8362      -0.0159 -0.0166 to -0.0151   27.47   -1.11          -0.13 -0.0162 -0.0126 -0.0248 -0.0165 -0.0152


In [4]:
training.show(training.gain_by_qp(results, manifest, base=results["bicubic"]))

                  model    0-18   18-26   26-30   30-34   34-38   38-42   42-46   46-52
                patches    3610    3928    3730    3585    3539    2846    2005    1043
                nearest -0.0156 -0.0166 -0.0173 -0.0172 -0.0166 -0.0149 -0.0126 -0.0107
               bilinear -0.0175 -0.0144 -0.0114 -0.0080 -0.0048 -0.0020 +0.0001 +0.0017
                bicubic +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000
bicubic, PIL (a = -0.5) -0.0020 -0.0016 -0.0011 -0.0007 -0.0003 +0.0002 +0.0004 +0.0007
         Lanczos-3, PIL +0.0023 +0.0016 +0.0010 +0.0005 +0.0000 -0.0004 -0.0005 -0.0006


## 3. Export for the browser

The three filters ONNX's `Resize` has, at the client's tile (320×180 LR pixels a step, plus each filter's halo: nearest
0, bilinear 1, bicubic 2) and as a whole 1080p frame, each checked against torch and timed on the desktop CPU.
onnxruntime's CPU `Resize` in cubic mode is slow; that is one kernel, not bicubic itself.

In [5]:
def input_shape(step_height, step_width, halo):
    """The model input for a step: the step plus the halo on every side, as (height, width)."""
    return (step_height + 2 * halo, step_width + 2 * halo)


def check_and_time(info, module, shape):
    """Check a written graph against the torch module, time it on the CPU, print one line."""
    webexport.check_graph(info["path"], module, np.random.rand(1, 3, *shape).astype(np.float32), np.float32, tolerance=2e-3)
    ms = webexport.time_graph(info["path"], shape, np.float32)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms))


tile_name = f"tile{STEP_WIDTH}x{STEP_HEIGHT}"
for mode, halo in (("nearest", 0), ("bilinear", 1), ("bicubic", 2)):
    module = models.Resize(SCALE, mode)
    tile_shape = input_shape(STEP_HEIGHT, STEP_WIDTH, halo)
    frame_shape = input_shape(FRAME_HEIGHT // SCALE, FRAME_WIDTH // SCALE, halo)
    tile = webexport.export(module, f"upscale_static_{mode}_{tile_name}.onnx", tile_shape,
                            label=f"static {mode} (Resize) fp32", halo=halo)
    check_and_time(tile, module, tile_shape)
    frame = webexport.to_frame(tile, f"upscale_static_{mode}_{tile_name}_frame.onnx", FRAME_WIDTH, FRAME_HEIGHT)
    check_and_time(frame, module, frame_shape)
print("compare in a browser: uv run src/upscale/benchmark/main.py")

upscale_static_nearest_tile320x180.onnx                 0.4 KB    0.09 ms   Constantx1, Resizex1   OUTSIDE BUDGET: Resize
upscale_static_nearest_tile320x180_frame.onnx           0.5 KB    2.10 ms   Constantx1, Resizex1   OUTSIDE BUDGET: Resize
upscale_static_bilinear_tile320x180.onnx                0.4 KB    0.59 ms   Constantx1, Resizex1   OUTSIDE BUDGET: Resize
upscale_static_bilinear_tile320x180_frame.onnx          0.5 KB    4.02 ms   Constantx1, Resizex1   OUTSIDE BUDGET: Resize
upscale_static_bicubic_tile320x180.onnx                 0.4 KB   49.21 ms   Constantx1, Resizex1   OUTSIDE BUDGET: Resize
upscale_static_bicubic_tile320x180_frame.onnx           0.5 KB  392.72 ms   Constantx1, Resizex1   OUTSIDE BUDGET: Resize
compare in a browser: uv run src/upscale/benchmark/main.py


## 4. What it says

**Bicubic is the bar: 0.8521 SSIM-Y, 28.58 dB of luma** on the 24,286 validation patches. No filter is far from it:

| filter | SSIM-Y | over bicubic | luma dB |
| --- | --- | --- | --- |
| Lanczos-3, PIL | 0.8528 | +0.0007 | 28.70 |
| **bicubic** (`models.bicubic`, a = −0.75) | **0.8521** | | **28.58** |
| bicubic, PIL (a = −0.5) | 0.8513 | −0.0008 | 28.50 |
| bilinear | 0.8434 | −0.0087 | 27.97 |
| nearest | 0.8362 | −0.0159 | 27.47 |

- **Lanczos-3 gains +0.0007**, on screens (+0.0013) and on clean frames (+0.0023 at QP 0–18), and loses from QP 38 up,
  where its sharper kernel brings out the codec's blocks; on video it is bicubic. ONNX cannot express it.
- **A softer bicubic is worth less on clean pictures and a little more on damaged ones**: PIL's (a = −0.5) loses
  0.0020 at QP 0–18 and gains 0.0007 at 46–52, and bilinear, softer still, reaches +0.0017 there. A blur hides blocks.
- **Nearest is worst everywhere**, most on photos (−0.0248).

So a learned model is measured against bicubic, the skip it is built on; a gain under about +0.001 is a filter's.

The graphs: nearest, bilinear and bicubic at the client's 320×180 step and as a whole 1080p frame. onnxruntime's CPU
`Resize` in cubic mode takes 38 ms a tile, a slow kernel the browser's WebGPU does not share.